In [ ]:
# AHN ditch detector
# import libraries
import pandas as pd
import rioxarray
import geopandas as gpd
import pyogrio
from shapely.geometry import box
import fiona
import numpy as np
import requests
import zipfile
from pathlib import Path
from urllib.parse import urlparse
from rasterstats import zonal_stats

In [ ]:
# Panda settings for showing data (this is foremost done to more easily explore the data while processing it)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
def download_file(url, output_folder=".", output_file=None):
    """
    Download a file from a URL.

    - If output_file is None, the filename is taken from the URL.
    - Files are saved in output_folder.
    - ZIP files are automatically extracted.
    - TIFF and GPKG files are left unchanged.

    Args:
        url (str): File URL
        output_folder (str): Folder to save the file in
        output_file (str, optional): Custom filename

    Returns:
        tuple:
            output_path (Path): Path to downloaded file
            to_be_deleted (Path): Path marked for deletion


    """

    response = requests.get(url)
    response.raise_for_status()

    # Create folder if it does not exist
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    # Use filename from URL if not provided
    if output_file is None:
        output_file = Path(urlparse(url).path).name

    output_path = output_folder / output_file

    # Store path in variable
    to_be_deleted = output_path

    # Save the downloaded file
    with open(output_path, "wb") as f:
        f.write(response.content)

    print(f"Downloaded: {output_path}")

    # Handle ZIP files
    if output_path.suffix.lower() == ".zip":
        extract_dir = output_folder / output_path.stem

        with zipfile.ZipFile(output_path, "r") as zip_ref:
            zip_ref.extractall(output_folder)

        print(f"Extracted ZIP to: {output_folder}")

        # Delete ZIP file after extraction
        output_path.unlink()

        print(f"Deleted ZIP file: {output_path}")

    # Handle TIFF files
    elif output_path.suffix.lower() in [".tif", ".tiff"]:
        print("TIFF file detected — no extraction performed.")

    # Handle GeoPackage files
    elif output_path.suffix.lower() == ".gpkg":
        print("GeoPackage file detected — no extraction performed.")

    else:
        print("File type not specially handled.")

    return output_path, to_be_deleted

In [ ]:
# Download bladwijzer geopackage AHN
download_file("https://basisdata.nl/hwh-ahn/AUX/bladwijzer.gpkg", "data//")

In [ ]:
bladwijzer = gpd.read_file('data//bladwijzer.gpkg')

In [ ]:
ahn_versie = "AHN3" #or AHN4 or AHN5

selection_AHN = bladwijzer[[ahn_versie+'_05M_M']]
#remove the none values from list
selection_AHN = selection_AHN.dropna(subset=[ahn_versie+'_05M_M'])

In [ ]:
brp_path = 'data//'
brp_name = 'brpgewaspercelen_definitief_2025'
brp_updated = brp_path+brp_name+"_updated.gpkg"

In [ ]:
print(len(selection_AHN[ahn_versie+"_05M_M"]))

In [ ]:
for dwn_link_ahn in selection_AHN["AHN3_05M_M"].iloc[:10]:

    output_path, to_be_deleted = download_file(dwn_link_ahn, "data//ahn_download//")

    ahn_tif_name = Path(urlparse(dwn_link_ahn).path).stem
    ahn_tif = rioxarray.open_rasterio("data//ahn_download//" + ahn_tif_name + ".TIF")

    ahn_tif = ahn_tif.squeeze()

    xres, yres = ahn_tif.rio.resolution()
    xres = abs(xres)
    yres = abs(yres)

    dz_dy, dz_dx = np.gradient(ahn_tif.values, yres, xres)

    slope_deg = np.degrees(
        np.arctan(
            np.sqrt(dz_dx**2 + dz_dy**2)
        )
    )

    slope = ahn_tif.copy(data=slope_deg.astype("float32"))
    slope.name = "slope_degrees"
    slope = slope.rio.write_crs(ahn_tif.rio.crs)

    xmin = slope.x.min().item()
    xmax = slope.x.max().item()
    ymin = slope.y.min().item()
    ymax = slope.y.max().item()
    bbox = (xmin, ymin, xmax, ymax)

    brp_section = pyogrio.read_dataframe(
        brp_updated,
        layer="brp_gewas",
        bbox=bbox
    )

    brp_section = brp_section[brp_section["gewascode"].isin([265, 266, 331])]

    if brp_section.empty:
        print(f"No BRP polygons found for {ahn_tif_name}, skipping zonal stats.")

    else:
        brp_clipped = gpd.clip(brp_section, bbox)

        transform = slope.rio.transform()

        stats = zonal_stats(
            brp_clipped,
            slope.values,
            affine=transform,
            stats=["sum"],
            nodata=slope.rio.nodata if hasattr(slope, "rio") else None,
        )

        brp_clipped["ras_sum_" + ahn_versie] = [s["sum"] for s in stats]

        stats_brp_result = brp_clipped[["ras_sum_" + ahn_versie, "uni_id"]]

        stats_brp_result.to_csv(
            "result_csv//" + ahn_versie + "_" + ahn_tif_name + ".csv",
            index=False
        )

    # cleanup
    ahn_tif.close()

    del slope
    del ahn_tif

    # sometimes needed on Windows
    import gc
    gc.collect()

    # delete downloaded dem
    to_be_deleted = to_be_deleted.with_suffix(".tif")

    file_path = Path(to_be_deleted)

    if file_path.exists():
        file_path.unlink()